# F-09 Whisper 파인튜닝 — Colab Pro+ (A100)

**실행 전 체크리스트**
- [ ] 런타임 유형: A100 GPU 선택
- [ ] Google Drive 마운트 확인

**실행 순서**: 셀 01 → 02 → 03 → 04 → 05 순서대로 실행
셀 04까지 완료 후 `런타임 > 백그라운드 실행 활성화` → 셀 05 실행

In [ ]:
# 셀 01 — 라이브러리 설치 + Drive 마운트 + 경로 설정
!pip install -q \
    transformers \
    datasets \
    peft \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    tensorboard

from google.colab import drive
from pathlib import Path
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH   = DRIVE_ROOT / 'processed/senior_speech'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/whisper-senior'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print('완료')
print(f'데이터셋 경로 : {DATASET_PATH}')
print(f'체크포인트 경로: {CHECKPOINT_DIR}')

In [ ]:
# 셀 02 — 전처리 실행 (Drive에 데이터 없을 때만)
import shutil
from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/Dadam_dataSet/processed/senior_speech')

if not DATASET_PATH.exists():
    shutil.copy('/content/drive/MyDrive/Dadam/whisper/preprocess.py', '/content/preprocess.py')
    !python /content/preprocess.py
    print('전처리 완료')
else:
    print(f'전처리 데이터 이미 존재 — 건너뜀: {DATASET_PATH}')

In [ ]:
# 셀 03 — 모델·프로세서 로드 + LoRA 설정 + 데이터셋 로드
import torch
from datasets import load_from_disk
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'openai/whisper-large-v3-turbo'

processor = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')

# float32로 로드 — fp16 변환은 Trainer가 자동 처리, 직접 float16 로드 시 평가 단계 dtype 충돌 발생
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

# generation_config 사용 — model.config 방식은 transformers 4.47+에서 ValueError 발생
model.generation_config.language = 'korean'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj'],
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

In [ ]:
# 셀 04 — Trainer 설정
from dataclasses import dataclass
from typing import Any
import numpy as np
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate

cer_metric = evaluate.load('cer')  # 한국어는 CER(음절 단위)이 WER보다 적합

EVAL_SAMPLES = 500  # validation 전체 사용 시 평가에만 수십 시간 소요 — 500개로 제한


@dataclass
class WhisperDataCollator:
    """배치별 패딩 처리 — 가변 길이 오디오를 30초로 패딩"""
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts = [f['text'] for f in features]

        # padding='max_length' + max_length=480000 — 미지정 시 짧은 오디오가 3000 프레임 미만으로 변환되어 오류 발생
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,        # 16000Hz × 30초 = 480000 샘플 → mel 3000 프레임
            truncation=True,
        )

        labels = self.processor.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=448,
        ).input_ids

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            'input_features': inputs.input_features,
            'labels': labels,
        }


def compute_metrics(pred):
    """CER 계산 — 검증 중 호출"""
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': round(cer, 4)}


training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,         # 유효 배치: 64
    learning_rate=1e-4,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',                 # evaluation_strategy는 구버전 파라미터 — eval_strategy 사용
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,                        # eval_steps와 동일해야 load_best_model_at_end 동작
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    report_to='tensorboard',
    save_total_limit=3,
    remove_unused_columns=False,           # DataCollator가 audio/text 직접 처리 — 제거 시 KeyError 발생
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'].select(range(EVAL_SAMPLES)),
    data_collator=WhisperDataCollator(processor=processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,  # transformers 4.47+에서 tokenizer 파라미터 → processing_class로 변경
)

print('Trainer 설정 완료 — 백그라운드 실행 활성화 후 셀 05 실행')

In [ ]:
# 셀 05 — 학습 시작
# 백그라운드 실행 활성화 후 실행할 것 (런타임 > 백그라운드 실행)
import os

last_checkpoint = None
if CHECKPOINT_DIR.exists():
    checkpoints = sorted(CHECKPOINT_DIR.glob('checkpoint-*'), key=os.path.getmtime)
    if checkpoints:
        last_checkpoint = str(checkpoints[-1])
        print(f'체크포인트 발견 — 이어서 학습: {last_checkpoint}')
    else:
        print('체크포인트 없음 — 처음부터 학습')
else:
    print('체크포인트 없음 — 처음부터 학습')

trainer.train(resume_from_checkpoint=last_checkpoint)

model.save_pretrained(str(CHECKPOINT_DIR / 'final'))
processor.save_pretrained(str(CHECKPOINT_DIR / 'final'))
print(f'학습 완료. 저장 위치: {CHECKPOINT_DIR}/final')

# 셀 06 — 추가 학습 (Fine-tune 2nd pass)

**목적**: 1차 학습(lr=1e-4, 4000 steps) 완료 후 CER이 목표(10%) 미달일 때 실행  
**방식**: `final/` 모델을 불러와 lr=1e-5로 2000 steps 추가 학습  
**실행 조건**: 셀 01(환경 설정)만 먼저 실행하면 됨 — 셀 03~05 재실행 불필요

In [ ]:
# 셀 07 — 추가 학습 (2nd pass: lr=1e-5, 2000 steps)
import os
import torch
import evaluate
from dataclasses import dataclass
from typing import Any
from pathlib import Path
from datasets import load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import PeftModel

# ── 경로 설정 ──────────────────────────────────────────────
DRIVE_ROOT      = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH    = DRIVE_ROOT / 'processed/senior_speech'
CHECKPOINT_DIR  = DRIVE_ROOT / 'checkpoints/whisper-senior'
FINAL_DIR       = CHECKPOINT_DIR / 'final'
FINAL2_DIR      = CHECKPOINT_DIR / 'final2'
CHECKPOINT2_DIR = CHECKPOINT_DIR / 'stage2'
CHECKPOINT2_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID     = 'openai/whisper-large-v3-turbo'
EVAL_SAMPLES = 500

# ── FINAL_DIR 존재 확인 ─────────────────────────────────────
# 1차 학습 미완료 상태에서 실행하면 PeftModel 로드 실패 — 사전 체크
if not FINAL_DIR.exists():
    raise FileNotFoundError(f'1차 학습 결과 없음: {FINAL_DIR}\n셀 05를 먼저 완료하세요.')

# ── 프로세서 로드 ───────────────────────────────────────────
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')

# ── 베이스 모델 + LoRA 가중치 로드 ─────────────────────────
# float32 로드 후 Trainer fp16=True로 변환 — 직접 float16 로드 시 평가 단계 dtype 충돌
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
base_model.generation_config.language = 'korean'
base_model.generation_config.task = 'transcribe'
base_model.generation_config.forced_decoder_ids = None
base_model.generation_config.suppress_tokens = []

# is_trainable=True — 추가 학습을 위해 LoRA 파라미터 그래디언트 활성화
model = PeftModel.from_pretrained(base_model, str(FINAL_DIR), is_trainable=True)
model.print_trainable_parameters()

# ── 데이터셋 로드 ───────────────────────────────────────────
dataset = load_from_disk(str(DATASET_PATH))

# ── DataCollator ────────────────────────────────────────────
@dataclass
class WhisperDataCollator:
    """배치별 패딩 처리 — 가변 길이 오디오를 30초로 패딩"""
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        labels = self.processor.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=448,
        ).input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'input_features': inputs.input_features, 'labels': labels}


# ── CER 메트릭 ──────────────────────────────────────────────
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {'cer': round(cer_metric.compute(predictions=pred_str, references=label_str), 4)}


# ── 2차 학습 설정 ───────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT2_DIR),
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,        # 1차(1e-4)의 1/10 — 미세 조정 단계
    warmup_steps=100,          # 2000 steps 기준 5% warmup
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,            # eval_steps와 동일해야 load_best_model_at_end 동작
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    report_to='tensorboard',
    save_total_limit=3,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'].select(range(EVAL_SAMPLES)),
    data_collator=WhisperDataCollator(processor=processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

# ── 체크포인트 이어받기 ─────────────────────────────────────
last_checkpoint = None
checkpoints = sorted(CHECKPOINT2_DIR.glob('checkpoint-*'), key=os.path.getmtime)
if checkpoints:
    last_checkpoint = str(checkpoints[-1])
    print(f'체크포인트 발견 — 이어서 학습: {last_checkpoint}')
else:
    print('체크포인트 없음 — 처음부터 2차 학습 시작')

trainer.train(resume_from_checkpoint=last_checkpoint)

# ── 최종 저장 ────────────────────────────────────────────────
model.save_pretrained(str(FINAL2_DIR))
processor.save_pretrained(str(FINAL2_DIR))
print(f'2차 학습 완료. 저장 위치: {FINAL2_DIR}')

In [ ]:
# 셀 08A — 검증 셋 샘플 10개 직접 확인
# 실행 전: 셀 01만 완료되면 됨
import re
import torch
from pathlib import Path
from datasets import load_from_disk
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

DRIVE_ROOT   = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH = DRIVE_ROOT / 'processed/senior_speech'
FINAL2_DIR   = DRIVE_ROOT / 'checkpoints/whisper-senior/final2'
MODEL_ID     = 'openai/whisper-large-v3-turbo'

PUNCT_PATTERN = re.compile(r'[.?!,。、]')

def clean_label(text: str) -> str:
    # SP 태그: 실제 단어로 교체 — (SP: 삼) → 사람, (SP:) → 제거
    # FP/기타 태그: 내용 제거 (채움말·잡음 표시)
    text = re.sub(r'\(SP:\s*([^)]+)\)', lambda m: m.group(1), text)  # (SP: 내용) → 내용으로 교체
    text = re.sub(r'\([A-Z]+:[^)]*\)', '', text)                      # 나머지 태그 제거
    text = re.sub(r'\[[^\]]*\]', '', text)                            # [...] 태그 제거
    text = PUNCT_PATTERN.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_pred(text: str) -> str:
    return PUNCT_PATTERN.sub('', text).strip()

processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
base_model.generation_config.language           = 'korean'
base_model.generation_config.task               = 'transcribe'
base_model.generation_config.forced_decoder_ids = None

model = PeftModel.from_pretrained(base_model, str(FINAL2_DIR), is_trainable=False)
model = model.to('cuda').eval()

dataset = load_from_disk(str(DATASET_PATH))

GEN_KWARGS = dict(language='korean', task='transcribe', num_beams=1)

print('=== 검증 셋 샘플 10개 확인 — SP 태그 교체 방식 ===\n')
for i in range(10):
    sample    = dataset['validation'][i]
    raw_label = sample['text']
    ref       = clean_label(raw_label)

    print(f'[{i:02d}] 원본 라벨  : {raw_label}')
    print(f'      정제 라벨  : {ref}')

    input_feats = processor(
        sample['audio']['array'],
        sampling_rate=16_000,
        return_tensors='pt',
    ).input_features.to('cuda').half()

    with torch.no_grad():
        pred_ids = model.generate(input_features=input_feats, **GEN_KWARGS)
    pred_text = clean_pred(processor.decode(pred_ids[0], skip_special_tokens=True))
    print(f'      모델 예측  : {pred_text}')
    print()

In [ ]:
# 셀 08B — 구두점+공백+태그 제거 CER로 500개 재평가
# 셀 08A 실행 후 model/processor/dataset/clean_label/clean_pred/GEN_KWARGS가 메모리에 있어야 함
import evaluate
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any

cer_metric   = evaluate.load('cer')
EVAL_SAMPLES = 500

@dataclass
class EvalCollator:
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        return {'input_features': inputs.input_features, 'texts': texts}

eval_subset = dataset['validation'].select(range(EVAL_SAMPLES))
loader      = DataLoader(eval_subset, batch_size=16, collate_fn=EvalCollator(processor))

all_preds, all_refs = [], []

for batch in loader:
    input_feats = batch['input_features'].to('cuda').half()
    with torch.no_grad():
        pred_ids = model.generate(input_features=input_feats, **GEN_KWARGS)

    preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
    refs  = batch['texts']

    # 구두점 + 공백 + 태그 모두 제거 후 음절 단위 CER
    all_preds.extend([clean_pred(p).replace(' ', '') for p in preds])
    all_refs.extend( [clean_label(r).replace(' ', '') for r in refs])

cer_result = cer_metric.compute(predictions=all_preds, references=all_refs)

print(f'구두점+태그 제거 CER ({EVAL_SAMPLES}개): {cer_result:.4f}  →  {cer_result * 100:.2f}%')
print()
print('── 예측 vs 정답 샘플 5개 ──')
for i in range(5):
    print(f'  정답: {all_refs[i]}')
    print(f'  예측: {all_preds[i]}')
    print()

In [ ]:
# 셀 09 — SP 태그 포함 샘플 제외 후 CER 재평가
# 셀 08A 실행 후 model/processor/dataset/clean_pred/GEN_KWARGS가 메모리에 있어야 함
import re
import evaluate
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any

SP_TAG = re.compile(r'\(SP:[^)]*\)')    # SP 태그 포함 여부 판별
PUNCT_PATTERN = re.compile(r'[.?!,。、]')

def clean_ref_no_sp(text: str) -> str:
    """SP 태그 없는 샘플용 — FP/기타 태그·구두점·공백만 제거"""
    text = re.sub(r'\([A-Z]+:[^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = PUNCT_PATTERN.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()

cer_metric   = evaluate.load('cer')
EVAL_SAMPLES = 500

val_data   = dataset['validation'].select(range(EVAL_SAMPLES))

# SP 태그 없는 샘플만 필터링
clean_indices = [i for i, ex in enumerate(val_data) if not SP_TAG.search(ex['text'])]
clean_subset  = val_data.select(clean_indices)
print(f'전체 {EVAL_SAMPLES}개 중 SP 태그 없는 샘플: {len(clean_subset)}개 ({len(clean_indices)/EVAL_SAMPLES*100:.1f}%)')

@dataclass
class EvalCollator:
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        return {'input_features': inputs.input_features, 'texts': texts}

loader = DataLoader(clean_subset, batch_size=16, collate_fn=EvalCollator(processor))

all_preds, all_refs = [], []

for batch in loader:
    input_feats = batch['input_features'].to('cuda').half()
    with torch.no_grad():
        pred_ids = model.generate(input_features=input_feats, **GEN_KWARGS)

    preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
    refs  = batch['texts']

    all_preds.extend([clean_pred(p).replace(' ', '') for p in preds])
    all_refs.extend( [clean_ref_no_sp(r).replace(' ', '') for r in refs])

cer_result = cer_metric.compute(predictions=all_preds, references=all_refs)

print(f'SP 태그 제외 CER ({len(clean_subset)}개): {cer_result:.4f}  →  {cer_result * 100:.2f}%')
print()
print('── 예측 vs 정답 샘플 5개 ──')
for i in range(min(5, len(all_refs))):
    print(f'  정답: {all_refs[i]}')
    print(f'  예측: {all_preds[i]}')
    print()